# Coco on a Colab GPU — train here, query from anywhere

**Notebook v1** · checked against
[`colab/VERSION`](https://github.com/saman-pasha/cocolog/blob/master/colab/VERSION)
when section 1 runs — the copy in your browser and the copy in the
repository are different things, and section 1 says so when they differ.
If it does: *File → Revert to saved*.

One VM trains on the GPU; the knowledge base survives in Google Drive;
a Cloudflare tunnel publishes a **read-only** view over Zeytun; any other
Colab, laptop toplevel or browser queries it with nothing but a URL.
[`COLAB.md`](https://github.com/saman-pasha/cocolog/blob/master/colab/COLAB.md)
beside this notebook explains the arrangement; this runs it.

**For training, pick a GPU runtime first**: *Runtime → Change runtime type →
GPU*. For a query-only session the default runtime is fine — run sections
1–2 and then the one-liner in section 9.

The build (section 1) takes a few minutes: ZiguratIP, then the transpile of
cocolog through Cicili, then the torch module against Colab's own PyTorch.

## 1 · Build all three

Cicili (the language, needed to transpile), ZiguratIP (the database), and
cocolog itself. Colab's preinstalled `torch` pip package provides libtorch —
headers, libraries and CUDA — so nothing torch-shaped is downloaded.

The cell checks the VM before it builds (`colab/preflight.sh`) and checks
the build by its artifacts afterwards (`colab/build.sh`) — ZiguratIP's
top-level `make` steps over a project that fails, so its exit code proves
nothing and the fourteen libraries are counted by name. If anything goes
wrong, the report names it; [COLAB.md's *When the build
fails*](https://github.com/saman-pasha/cocolog/blob/master/colab/COLAB.md#when-the-build-fails)
has the known ones.


In [ ]:
import os, subprocess

# THE VERSION OF THIS NOTEBOOK -- the copy running in your browser.
# Bumped in colab/VERSION whenever a cell changes; the check further
# down is what makes it worth having. Printed FIRST so that it travels
# with any output you paste at anyone: a bug report without a version
# is a question nobody can answer, and this notebook has already been
# on both sides of that.
NOTEBOOK_VERSION = 1

print(f'== Coco Colab notebook v{NOTEBOOK_VERSION}')

CICILI_REPO   = 'https://github.com/saman-pasha/cicili.git'
ZIGURAT_REPO  = 'https://github.com/saman-pasha/ZiguratIP.git'
COCOLOG_REPO  = 'https://github.com/saman-pasha/cocolog.git'

# THE PREREQUISITES ARE INSTALLED FROM THE REPO, not from this cell --
# see colab/prereqs.sh. A notebook cell is a copy of a fact that lives
# in the repository, and this one drifted once already: the package list
# was corrected upstream, the cell in the browser stayed as it was, and
# the next run installed the wrong thing and refused twice for the same
# reason. So the repos are cloned FIRST and the list comes with them.

for repo, path in [(CICILI_REPO, '/content/cicili'),
                   (ZIGURAT_REPO, '/content/ZiguratIP'),
                   (COCOLOG_REPO, '/content/cocolog')]:
    if not os.path.isdir(path):
        subprocess.run(['git', 'clone', '--depth', '1', repo, path], check=True)
    else:
        subprocess.run(['git', '-C', path, 'fetch', '--depth', '1', 'origin'], check=True)
        subprocess.run(['git', '-C', path, 'reset', '--hard', 'origin/master'], check=True)

# WHICH VERSION AM I ACTUALLY RUNNING? Two answers, and they can
# differ. The repository is up to date -- that reset --hard just made it
# so -- but THIS CELL is whatever your browser last loaded, and a
# browser holding a stale notebook is precisely how the same failure
# arrived twice with its fix already sitting on disk. So the two are
# compared out loud, and the three commits printed beside them.
#
# It is a WARNING and not a refusal. The scripts come from the
# repository, which is the whole point of their being scripts, so an
# older notebook usually still builds -- and stopping a working build
# over a number would be the worse mistake. Same judgement as the
# preflight's ABI check, for the same reason.
try:
    repo_version = open('/content/cocolog/colab/VERSION').readline().strip()
except OSError:
    repo_version = ''

heads = {name: subprocess.run(['git', '-C', p, 'rev-parse', '--short', 'HEAD'],
                              capture_output=True, text=True).stdout.strip()
         for name, p in [('cicili', '/content/cicili'),
                         ('ZiguratIP', '/content/ZiguratIP'),
                         ('cocolog', '/content/cocolog')]}
print('   repo colab/VERSION  v%s' % (repo_version or '?'))
print('   commits             ' + '  '.join(f'{k} {v}' for k, v in heads.items()))

if repo_version and repo_version != str(NOTEBOOK_VERSION):
    print()
    print('   ' + '!' * 66)
    print(f'   THIS NOTEBOOK IS v{NOTEBOOK_VERSION}; THE REPOSITORY IS AT v{repo_version}.')
    print('   Your browser is holding an older copy of the cells. The files on')
    print('   disk are current -- git reset --hard saw to that -- so the build')
    print('   will probably still work. But if it fails for a reason you have')
    print('   already fixed, THIS IS WHY: File -> Revert to saved, then re-run.')
    print('   ' + '!' * 66)
print()

pre = subprocess.run(['sh', 'colab/prereqs.sh'], cwd='/content/cocolog')
if pre.returncode != 0:
    raise SystemExit('the prerequisites would not install -- see the output above')

os.environ['CICILI']          = '/content/cicili'
os.environ['ZIGURATIP']       = '/content/ZiguratIP'
os.environ['ZIGURATIP_HOME']  = '/content/ZiguratIP/home'
os.environ['COCOLOG']         = '/content/cocolog'
os.environ['LD_LIBRARY_PATH'] = os.environ['ZIGURATIP_HOME'] + '/lib'

# What this VM actually has -- every tool with its version, and libtorch
# with the C++ ABI its wheel was built against. A missing one names the
# package that carries it rather than failing later as a linker error.
# The report is CAPTURED as well as shown, so that if it refuses, the
# lines that say why travel with the exception. The first version of
# this cell threw them away and left an assertion that knew nothing.
pre = subprocess.run(['sh', 'colab/preflight.sh'], cwd='/content/cocolog',
                     capture_output=True, text=True)
print(pre.stdout, end='')
if pre.stderr: print(pre.stderr, end='')
if pre.returncode != 0:
    named = [l for l in pre.stdout.splitlines() if 'MISSING' in l or 'RED' in l]
    raise SystemExit('preflight RED:\n  ' + '\n  '.join(named or ['(see the report above)']))

# The build. Both makes are checked by ARTIFACT and not by exit code --
# ZiguratIP's top-level make steps over a project that fails -- and a
# failure prints the lines that say why. Takes a few minutes.
out = subprocess.run(['sh', 'colab/build.sh'], cwd='/content/cocolog')
if out.returncode != 0:
    raise SystemExit('build RED -- the report above names the cause')


## 2 · Does the torch module see the GPU?

`torch_device(auto)` takes `cuda` when it is available, so this is the
whole GPU story. `false` here means a CPU runtime — training still works,
slower.

In [ ]:
!nvidia-smi -L 2>/dev/null || echo '(no GPU runtime)'
!./cocolog query "torch_cuda_available(B)"
!./cocolog query "torch_device(auto)" 

## 3 · (Optional) restore the knowledge base from Google Drive

Colab VMs are ephemeral. `PERSIST = True` copies a snapshot from Drive onto
**local disk** before the server starts — the engine never runs against the
Drive FUSE mount. Three directories travel together: `data` (the rows),
`catalog` (what links against what), `ld` (compiled objects); the MANIFEST
says which builds they came from, so a mismatch is reported rather than
crashing later with an undefined symbol.

In [ ]:
PERSIST = False  # set True to keep the knowledge base across sessions via Drive
DRIVE_BACKUP  = '/content/drive/MyDrive/coco-kb'
SNAPSHOT_DIRS = ['data', 'catalog', 'ld']

def commit_of(path):
    done = subprocess.run(['git', '-C', path, 'rev-parse', 'HEAD'],
                          capture_output=True, text=True)
    return done.stdout.strip() if done.returncode == 0 else ''

if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(DRIVE_BACKUP):
        print('no snapshot in', DRIVE_BACKUP, 'yet -- a fresh store on first use.')
    else:
        restored = []
        for name in SNAPSHOT_DIRS:
            source = os.path.join(DRIVE_BACKUP, name)
            if not os.path.isdir(source):
                continue
            target = os.path.join(os.environ['ZIGURATIP_HOME'], name)
            subprocess.run(['rm', '-rf', target], check=True)
            subprocess.run(['cp', '-a', source, target], check=True)
            restored.append(name)
        print('restored:', ', '.join(restored) or 'nothing')
        manifest = os.path.join(DRIVE_BACKUP, 'MANIFEST')
        if os.path.exists(manifest):
            saved = dict(line.strip().split(': ', 1) for line in open(manifest)
                         if ': ' in line)
            for repo, path in [('ziguratip', '/content/ZiguratIP'),
                               ('cocolog', '/content/cocolog')]:
                if saved.get(repo) and saved[repo] != commit_of(path):
                    print(f'SNAPSHOT IS FROM A DIFFERENT {repo} BUILD',
                          saved[repo][:12], 'vs', commit_of(path)[:12],
                          '-- if a page fails to load, re-run make schema.')
else:
    print('PERSIST is off -- a fresh store is created on first use.')

## 4 · Start the server, and check both ports

The trainer talks to 2160 (the binary protocol, loopback only). Zeytun on
2190 is what the tunnel will publish.

In [ ]:
import time, urllib.request

try:
    srv.terminate(); srv.wait(timeout=5)   # a previous instance, on re-run
except Exception:
    pass

%cd /content/ZiguratIP
srv = subprocess.Popen(['./home/bin/ziguratip'],
                       stdout=open('/content/server.log', 'w'),
                       stderr=subprocess.STDOUT, env=os.environ)
time.sleep(2.5)
assert srv.poll() is None, 'server exited -- see /content/server.log'

for _ in range(10):
    try:
        with urllib.request.urlopen('http://127.0.0.1:2190/', timeout=5) as r:
            print('Zeytun answers: HTTP', r.status)
        break
    except Exception:
        time.sleep(1)

%cd /content/cocolog
!./cocolog --kb brain list

## 5 · Train, on the GPU, into the knowledge base

Any tutorial works this way; XOR is the small one. `torch_device(auto)`
runs the training on the GPU when there is one, and `model_save/2` inside
`train` puts the model into the knowledge base **as clauses** —
`model_spec/2` and `model_params/2` — which is what makes it queryable
from everywhere else. Tensors come back to the CPU before they reach the
store, so what is saved is device-free.

In [ ]:
!./cocolog --kb brain run tutorials/07-xor.pl "torch_device(auto), train"
!./cocolog --kb brain run tutorials/07-xor.pl test

## 6 · A public URL, through Cloudflare

**Off by default** — running a notebook top to bottom should not put a
server on the internet as a side effect. Set `TUNNEL = True` and read
[ZiguratIP's warning](https://github.com/saman-pasha/ZiguratIP/blob/master/colab/TUTORIAL.md)
first: a quick tunnel has **no authentication** — anyone with the URL reads
every knowledge base this server holds — and the cell refuses to start
while a Parsi compiler page is loadable, because that would be a compiler
behind an HTTP form.

What the tunnel publishes is Zeytun, whose knowledge-base backend fills
only the read hooks — a querier **cannot write** through it, by
construction. The writer stays on this VM's loopback.

In [ ]:
import re, glob

TUNNEL       = False   # set True to publish the read-only view
I_UNDERSTAND = False   # override the compiler-page check below (do not)

CF, LOG = '/content/cloudflared', '/content/cloudflared.log'
HOME = os.environ['ZIGURATIP_HOME']

def unsafe_to_expose():
    reasons = []
    conf = os.path.join(HOME, 'etc', 'ziguratip.conf')
    try:
        for line in open(conf):
            bare = line.split('#', 1)[0]
            if 'REMOTE_MODE' in bare and 'TRUE' in bare.upper():
                reasons.append('COMPILER/REMOTE_MODE is TRUE in ' + conf)
                break
    except FileNotFoundError:
        pass
    for so in sorted(glob.glob(os.path.join(HOME, 'ld', 'lib_*COMPILER*_.so'))):
        reasons.append(so + ' is loadable -- a compiler page must not be published')
    return reasons

if not TUNNEL:
    print('TUNNEL is off. Nothing is exposed.')
else:
    problems = unsafe_to_expose()
    if problems and not I_UNDERSTAND:
        print('REFUSING TO OPEN A TUNNEL. Found:')
        for p in problems: print('  -', p)
    else:
        if problems:
            print('WARNING -- opening anyway because I_UNDERSTAND is set')
        if not os.path.exists(CF):
            subprocess.run(['curl', '-sSL', '-o', CF,
                'https://github.com/cloudflare/cloudflared/releases/latest/'
                'download/cloudflared-linux-amd64'], check=True)
            os.chmod(CF, 0o755)
        old = globals().get('_tunnel')
        if old is not None and old.poll() is None:
            old.terminate()
        with open(LOG, 'w') as log:
            _tunnel = subprocess.Popen(
                [CF, 'tunnel', '--url', 'http://localhost:2190', '--no-autoupdate'],
                stdout=log, stderr=subprocess.STDOUT)
        globals()['_tunnel'] = _tunnel
        url = None
        for _ in range(45):
            if _tunnel.poll() is not None:
                break
            found = re.findall(r'https://[a-z0-9][a-z0-9-]*\.trycloudflare\.com',
                               open(LOG).read())
            if found:
                url = found[0]; break
            time.sleep(2)
        if url:
            name = url[len('https://'):]
            print('Browser        :', url)
            print()
            print('From any cocolog -- another Colab, a laptop toplevel:')
            print()
            print('  cocolog --host %s --http 80 --kb brain' % name)
            print()
            print("  ?- model_load(xor, M), model_predict(M, [[0.0,1.0]], P).")
            print()
            print('Stop it with:  _tunnel.terminate()')
        else:
            print('NO TUNNEL URL. cloudflared said:')
            for line in open(LOG).read().splitlines()[-10:]:
                print('  ', line[:160])

## 7 · The same query, from this VM

What a remote querier will see, rehearsed over loopback: the model loads
from the knowledge base as terms and predicts on whatever device the
querier has — here, the same machine; on a laptop, its CPU.

In [ ]:
!./cocolog --host 127.0.0.1 --http 2190 --kb brain \
  query "model_load(xor, M), model_predict(M, [[0.0,1.0],[1.0,1.0]], P)" 

## 8 · Query it from anywhere else

* **Another Colab** — run sections 1–2 of this notebook there (default
  runtime; no server, no tunnel, no GPU needed), then the
  `--http 80` one-liner section 6 printed.
* **Your own machine** — a built cocolog and the URL are enough; the
  toplevel runs in the `--http` arrangement like any other:

```
$ cocolog --host NAME.trycloudflare.com --http 80 --kb brain
?- model_load(xor, M), model_predict(M, [[0.0,1.0]], P).
```

* **A browser** — the `https://` URL serves Zeytun's pages directly.

Reads only: `assertz/1` from a querier fails (Zeytun has no write hook),
and machine claiming needs the binary port, which never crosses the
tunnel. Expect tunnel latency — the knowledge base is fetched a page per
predicate, each its own round trip; `--timeout 60` helps on a slow link.

## 9 · Snapshot the knowledge base to Drive

Only after a clean stop — there is no write-ahead log, so a snapshot of a
running server can be torn, and this copy is what the next session
restores.

In [ ]:
_srv = globals().get('srv')
if not PERSIST:
    print('PERSIST is off -- nothing to save.')
elif _srv is not None and _srv.poll() is None:
    print('THE SERVER IS STILL RUNNING. Stop it (next cell), then run this.')
else:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    saved = []
    for name in SNAPSHOT_DIRS:
        source = os.path.join(os.environ['ZIGURATIP_HOME'], name)
        if not os.path.isdir(source):
            continue
        target = os.path.join(DRIVE_BACKUP, name)
        subprocess.run(['rm', '-rf', target], check=True)
        subprocess.run(['cp', '-a', source, target], check=True)
        saved.append(name)
    with open(os.path.join(DRIVE_BACKUP, 'MANIFEST'), 'w') as f:
        f.write('ziguratip: %s\n' % commit_of('/content/ZiguratIP'))
        f.write('cocolog: %s\n' % commit_of('/content/cocolog'))
        f.write('dirs: %s\n' % ' '.join(saved))
    print('snapshot written to', DRIVE_BACKUP, '--', ', '.join(saved))

## 10 · Stop

In [ ]:
t = globals().get('_tunnel')
if t is not None and t.poll() is None:
    t.terminate()
    print('tunnel stopped')
_srv = globals().get('srv')
if _srv is not None and _srv.poll() is None:
    _srv.terminate()
    try:
        _srv.wait(timeout=10)
    except Exception:
        _srv.kill()
    print('server stopped -- now the snapshot cell may run')